In [ ]:
# ===============================
# 1️⃣ Imports
# ===============================
import pandas as pd
import numpy as np
import torch
import transformers
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import json

In [ ]:
import torch
import transformers

print(torch.__version__)
print(transformers.__version__)


In [ ]:
# ===============================
# 2️⃣ Charger les données JSONL
# ===============================
def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

train_df = load_jsonl("../../data/train.jsonl")
test_df = load_jsonl("../../data/kaggle_test.jsonl")

print(train_df.head())
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# Suppose que la colonne de texte s'appelle 'text' et le label 'label'
# Vérifie les noms exacts dans ton fichier train.jsonl !
TEXT_COL = "text"
LABEL_COL = "label"


In [ ]:
# ===============================
# 3️⃣ Split train/validation
# ===============================
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df[TEXT_COL].tolist(),
    train_df[LABEL_COL].tolist(),
    test_size=0.1,
    random_state=42,
    stratify=train_df[LABEL_COL]
)

In [ ]:
# ===============================
# 4️⃣ Tokenization avec BERTweet
# ===============================
model_name = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_df[TEXT_COL].tolist(), truncation=True, padding=True, max_length=128)

In [ ]:
# ===============================
# 5️⃣ Dataset PyTorch
# ===============================
class TweetDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

train_dataset = TweetDataset(train_encodings, train_labels)
val_dataset = TweetDataset(val_encodings, val_labels)
test_dataset = TweetDataset(test_encodings)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

# ===============================
# 6️⃣ Charger le modèle
# ===============================
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


In [ ]:
# ===============================
# 7️⃣ Entraînement
# ===============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(torch.cuda.is_available())

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:

# ===============================
# 8️⃣ Évaluation finale
# ===============================
metrics = trainer.evaluate()
print("Validation accuracy:", metrics["eval_accuracy"])

# ===============================
# 9️⃣ Prédictions sur le test set
# ===============================
preds = trainer.predict(test_dataset)
test_preds = np.argmax(preds.predictions, axis=1)

submission = pd.DataFrame({
    "ID": test_df["ID"],  # colonne à adapter selon ton fichier
    "Prediction": test_preds
})
submission.to_csv("submission.csv", index=False)
print("✅ Fichier submission.csv généré !")
